# Notebook to open `xarray` datasets on SciServer-ceph and display basic information about them.

This code avoids OceanSpy

If the datasets fail to open, make sure that your SciServer container includes the Poseidon (ceph) and Ocean Circulation (ceph) data volumes.

TWNH Jun '26

In [1]:
import intake
import traceback

In [2]:
# Change this to the catalog you want to test
local_catalog_file1 = '/home/idies/workspace/Storage/Thomas.Haine/persistent/Poseidon testing/ceph-dev/oceanspy/sciserver_catalogs/catalog_xarray.yaml'

# If False: read one scalar from the first data variable only.
# If True: read one scalar from every data variable.
CHECK_ALL_VARS = False

In [3]:
def light_check(ds, check_all_vars=False):
    """
    Light integrity check for an xarray Dataset.

    This does not load the whole dataset. It only reads one scalar from one
    variable, or from every variable if check_all_vars=True.
    """

    print(f"    dims: {dict(ds.sizes)}")
    print(f"    data variables: {len(ds.data_vars)}")

    if len(ds.data_vars) == 0:
        print("    no data variables to test")
        return

    varnames = list(ds.data_vars) if check_all_vars else [list(ds.data_vars)[0]]

    for varname in varnames:
        da = ds[varname]

        print(f"    checking variable: {varname}, dims={da.dims}, shape={da.shape}")

        if da.size == 0:
            print("      skipping empty variable")
            continue

        indexer = {
            dim: 0
            for dim in da.dims
            if da.sizes.get(dim, 0) > 0
        }

        # Trigger a tiny actual read.
        da.isel(indexer).load()

        print("      ok")
        
def test_catalog(catalog_path, check_all_vars=False):
    """
    Open every source in an Intake catalog and run a light xarray integrity check.
    """

    cat = intake.open_catalog(catalog_path)

    passed = []
    failed = []

    for name in cat:
        print("\n" + "=" * 80)
        print(f"Testing source: {name}")
        print("=" * 80)

        try:
            source = cat[name]

            # Intake opens it as an xarray Dataset, usually Dask-backed.
            ds = source.to_dask()

            light_check(ds, check_all_vars=check_all_vars)

            try:
                ds.close()
            except Exception:
                pass

            print(f"PASS: {name}")
            passed.append(name)

        except Exception as e:
            print(f"FAIL: {name}")
            print(f"Error: {e}")
            traceback.print_exc()
            failed.append((name, repr(e)))

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Catalog: {catalog_path}")
    print(f"Passed: {len(passed)}")
    print(f"Failed: {len(failed)}")

    if failed:
        print("\nFailed sources:")
        for name, error in failed:
            print(f"  - {name}: {error}")

    return passed, failed

In [4]:
passed, failed = test_catalog(
    local_catalog_file1,
    check_all_vars=CHECK_ALL_VARS,
)


Testing source: grd_get_started
    dims: {'Z': 216, 'Zp1': 217, 'Zu': 216, 'Zl': 216, 'X': 960, 'Y': 880, 'Xp1': 961, 'Yp1': 881}
    data variables: 30
    checking variable: drC, dims=('Zp1',), shape=(217,)
      ok
PASS: grd_get_started

Testing source: fld_get_started
    dims: {'X': 960, 'Y': 880, 'Xp1': 961, 'Yp1': 881, 'Z': 216, 'Zl': 216, 'T': 4, 'Zd000001': 1, 'Zmd000216': 216}
    data variables: 49
    checking variable: EXFhs, dims=('T', 'Zd000001', 'Y', 'X'), shape=(4, 1, 880, 960)
      ok
PASS: fld_get_started

Testing source: avg_get_started
    dims: {'X': 207, 'Xp1': 208, 'Y': 154, 'Yp1': 155, 'T': 4, 'Zld000216': 55, 'Zmd000216': 55, 'Zd000001': 1}
    data variables: 14
    checking variable: ADVr_SLT, dims=('T', 'Zld000216', 'Y', 'X'), shape=(4, 55, 154, 207)
      ok
PASS: avg_get_started

Testing source: grd_IGPwinter
    dims: {'time_midp': 359, 'Zl': 216, 'Y': 880, 'X': 960, 'Z': 216, 'Xp1': 961, 'Yp1': 881, 'time': 360, 'Zp1': 217, 'Zu': 216}
    data variab

Traceback (most recent call last):
  File "/tmp/ipykernel_6579/2464483274.py", line 57, in test_catalog
    ds = source.to_dask()
  File "/home/idies/mambaforge/envs/Oceanography/lib/python3.10/site-packages/intake_xarray/base.py", line 69, in to_dask
    return self.read_chunked()
  File "/home/idies/mambaforge/envs/Oceanography/lib/python3.10/site-packages/intake_xarray/base.py", line 44, in read_chunked
    self._load_metadata()
  File "/home/idies/mambaforge/envs/Oceanography/lib/python3.10/site-packages/intake/source/base.py", line 283, in _load_metadata
    self._schema = self._get_schema()
  File "/home/idies/mambaforge/envs/Oceanography/lib/python3.10/site-packages/intake_xarray/base.py", line 18, in _get_schema
    self._open_dataset()
  File "/home/idies/mambaforge/envs/Oceanography/lib/python3.10/site-packages/intake_xarray/xzarr.py", line 44, in _open_dataset
    self._ds = xr.open_mfdataset(self.urlpath, **kw)
  File "/home/idies/mambaforge/envs/Oceanography/lib/python3.10


Testing source: ECCO_v4r4
    dims: {'time': 312, 'k_l': 50, 'face': 13, 'j': 90, 'i': 90, 'k': 50, 'i_g': 90, 'j_g': 90, 'time_snap': 311, 'k_p1': 51, 'k_u': 50, 'nv': 2}
    data variables: 37
    checking variable: ADVr_SLT, dims=('time', 'k_l', 'face', 'j', 'i'), shape=(312, 50, 13, 90, 90)
      ok
PASS: ECCO_v4r4

Testing source: daily_ecco_grid
    dims: {'k_p1': 51, 'j_g': 90, 'i_g': 90, 'k': 50, 'j': 90, 'k_u': 50, 'i': 90, 'k_l': 50, 'tile': 13}
    data variables: 0
    no data variables to test
PASS: daily_ecco_grid

Testing source: daily_ecco_snap
    dims: {'time_midp': 9496, 'face': 13, 'Y': 90, 'X': 90, 'Z': 50}
    data variables: 4
    checking variable: ETAN_snap, dims=('time_midp', 'face', 'Y', 'X'), shape=(9496, 13, 90, 90)
      ok
PASS: daily_ecco_snap

Testing source: daily_ecco_mean
    dims: {'time': 9497, 'Zl': 50, 'face': 13, 'Y': 90, 'X': 90, 'Z': 50, 'Xp1': 90, 'Yp1': 90}
    data variables: 31
    checking variable: ADVr_SLT, dims=('time', 'Zl', 'face', 

In [5]:
import pandas as pd

results = pd.DataFrame(
    [{"source": name, "status": "passed", "error": ""} for name in passed]
    +
    [{"source": name, "status": "failed", "error": error} for name, error in failed]
)

results

,source,status,error
0,grd_get_started,passed,
1,fld_get_started,passed,
2,avg_get_started,passed,
3,grd_IGPwinter,passed,
4,fld_IGPwinter,passed,
5,grd_IGPyearlong,passed,
6,fld_IGPyearlong,passed,
7,grd_EGshelfIIseas2km_ERAI_6H,passed,
8,fld_EGshelfIIseas2km_ERAI_6H,passed,
9,grd_EGshelfIIseas2km_ERAI_1D,passed,
